# CRIVO — Binding de Identidade (GPU)

Treino focado no gargalo confirmado pela auditoria rígida: preservar o nome exato da entidade enquanto mantém categoria, função, correção e invariância.

Esta versão limpa processos antigos, copia o checkpoint-base para o SSD local, faz um pré-teste de carregamento/GPU e só então inicia os 8.000 passos. Nada é ativado no chat automaticamente.


In [ ]:
!nvidia-smi
!pip -q install 'tokenizers>=0.20,<1'
import torch
from google.colab import drive
assert torch.cuda.is_available(), 'Ative GPU em Ambiente de execução > Alterar tipo de ambiente > GPU.'
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')


In [ ]:
%cd /content
!rm -rf CRIVO
!git clone -q --branch colab/treino-binding-identidade-20261012 --single-branch https://github.com/HROSONE/CRIVO.git
%cd /content/CRIVO
!git rev-parse HEAD


## Treino robusto

Esta célula mata qualquer processo antigo desse treino, usa o SSD do Colab e mostra um pré-check antes de começar. Depois aparecem `blind_before 100/500` e os marcos `250/2000`.


In [ ]:
import os, sys, shutil, subprocess, signal, time
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/CRIVO/checkpoints')
LOCAL_ROOT = Path('/content/crivo_binding_cache')

print('🧹 Encerrando processos antigos deste treino...')
subprocess.run(['bash','-lc', "pkill -f '[t]reino_binding_identidade.py' || true"], check=False)
time.sleep(2)

curricular = DRIVE_ROOT / 'escola_curricular_longa'
runs = sorted(p for p in curricular.glob('run_*') if (p/'05_05_dialogo_estado'/'pesos.pt').exists())
assert runs, '❌ Não encontrei a fase 5 do treino curricular no Drive.'
drive_run = runs[-1]
drive_base = drive_run / '05_05_dialogo_estado'
local_base = LOCAL_ROOT / 'escola_curricular_longa' / drive_run.name / '05_05_dialogo_estado'

if LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT)
local_base.mkdir(parents=True, exist_ok=True)

print('✅ Checkpoint encontrado:', drive_base)
for nome in ('pesos.pt','tokenizer.json'):
    print('📥 Copiando', nome, 'para SSD local...', flush=True)
    shutil.copy2(drive_base/nome, local_base/nome)
print('✅ Checkpoint copiado para o SSD local.')

env = os.environ.copy()
env['PYTHONPATH'] = '/content/CRIVO'
env['PYTHONHASHSEED'] = '0'
env['PYTHONUNBUFFERED'] = '1'

probe = f'''\nimport time, torch\nfrom tokenizers import Tokenizer\nfrom linguagem_profunda import Configuracao, LinguagemProfunda\np=r"{local_base}"\nprint('[PRECHECK 1/4] lendo pesos...', flush=True)\nt=time.time(); s=torch.load(p+'/pesos.pt', map_location='cpu', weights_only=True); print('[PRECHECK 1/4] pesos OK', round(time.time()-t,2),'s', flush=True)\nprint('[PRECHECK 2/4] tokenizer...', flush=True); Tokenizer.from_file(p+'/tokenizer.json'); print('[PRECHECK 2/4] tokenizer OK', flush=True)\nprint('[PRECHECK 3/4] montando modelo...', flush=True); m=LinguagemProfunda(Configuracao(**s['config'])); m.load_state_dict(s['modelo'], strict=True); print('[PRECHECK 3/4] modelo OK', flush=True)\nprint('[PRECHECK 4/4] movendo para GPU...', flush=True); m=m.cuda(); torch.cuda.synchronize(); print('[PRECHECK 4/4] GPU OK', round(torch.cuda.memory_allocated()/1024**2,1),'MiB', flush=True)\n'''
print('\n🔎 Fazendo pré-check completo...')
pre = subprocess.run([sys.executable,'-u','-c',probe], cwd='/content/CRIVO', env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=120)
print(pre.stdout, end='')
assert pre.returncode == 0, '❌ Pré-check falhou; o treino NÃO foi iniciado.'

print('\n🚀 INICIANDO TREINO REAL NA GPU...')
cmd=[sys.executable,'-u','/content/CRIVO/scripts/treino_binding_identidade.py','--drive-root',str(LOCAL_ROOT),'--batch-size','16']
p=subprocess.Popen(cmd,cwd='/content/CRIVO',env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
try:
    for line in p.stdout:
        print(line,end='',flush=True)
    rc=p.wait()
except KeyboardInterrupt:
    print('\n⛔ Interrompido: encerrando também o processo filho para não deixar a GPU presa...')
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
    except Exception:
        p.terminate()
    raise
assert rc == 0, f'❌ Treino falhou com código {rc}'

local_output = LOCAL_ROOT / 'escola_binding_identidade'
drive_output = DRIVE_ROOT / 'escola_binding_identidade'
assert local_output.exists(), '❌ Treino terminou sem pasta de saída.'
drive_output.mkdir(parents=True, exist_ok=True)
print('\n📤 Copiando resultados para o Google Drive...')
shutil.copytree(local_output, drive_output, dirs_exist_ok=True)
print('✅ RESULTADOS SALVOS NO DRIVE:', drive_output)
print('✅ TREINO CONCLUÍDO E PROCESSO LIMPO.')


Os resultados são salvos em `Meu Drive/CRIVO/checkpoints/escola_binding_identidade/`. O checkpoint permanece `ativo_no_chat=false`.
